# GI Mod Analyzer
[![Static Badge](https://img.shields.io/badge/Jupyter_Notebook-F37726?style=for-the-badge)](https://jupyter.org/)

<br>

Gives a basic, high level summary of a character mod for the game GI

<br>

## Contributors

|   |   |
|---|---|
| **[Albert Gold](https://github.com/Alex-Au1)** | [![](https://dcbadge.limes.pink/api/shield/367087171154214914?theme=discord-inverted)](https://discordlookup.com/user/367087171154214914) |

<br>

## Requirements
- Python (Version 3.6 or up)

<br>
<br>

## Installation
Choose how to install AGRemap's API

**Option A**: If you want to install through [Pypi](https://pypi.org/project/AnimeGameRemap/), you run the pip install command below

In [ ]:
%pip install -U AnimeGameRemap

In [ ]:
import AnimeGameRemap as AGR

<br>

**Option B**: Alternatively, you can locally import the API from a specific git branch

In [2]:
import sys

# Note: Make sure the path correctly points where the AGRemap's API is located
sys.path.insert(1, r"../../../Anime Game Remap (for all users)/api")

import src.FixRaidenBoss2 as AGR

<br>
<br>

## Initialization
Run the codeblock below to initialize the necessary tools for the analyzing the mods

In [3]:
from enum import Enum
from typing import Dict, List


class FileTypes(Enum):
    VB = "vb"
    IB = "ib"


class StrClassifiers(Enum):
    FileType = AGR.AhoCorasickBuilder().build(data = {"Blend.buf": (FileTypes.VB, AGR.IniKeywords.Blend),
                                                    "Texcoord.buf": (FileTypes.VB, AGR.IniKeywords.Texcoord),
                                                    "Position.buf": (FileTypes.VB, AGR.IniKeywords.Position),
                                                    "Head.ib": (FileTypes.IB, "head"),
                                                    "Body.ib": (FileTypes.IB, "body"),
                                                    "Dress.ib": (FileTypes.IB, "dress"),
                                                    "Extra.ib": (FileTypes.IB, "extra")})

class ModAnalyzer():
    def __init__(self, folder: str):
        self.folder = folder
        self.vbFiles: Dict[str, str] = {}
        self.ibFiles: Dict[str, str] = {}

        self.vbFileContent: Dict[str, bytes] = {}
        self.ibFileContent: Dict[str, bytes] = {}

        self.numOfVertices = 0
        self.numOfTriangles: Dict[str, int] = {}
        self.vbSizePerVertex: Dict[str, int] = {}

    def groupFiles(self, files: List[str]):
        for file in files:
            keyword, fileClassification = StrClassifiers.FileType.value.getMaximal(file, errorOnNotFound = False)
            if (fileClassification is None):
                continue

            fileType, fileSubType = fileClassification

            if (fileType == FileTypes.VB):
                self.vbFiles[fileSubType] = file
            elif (fileType == FileTypes.IB):
                self.ibFiles[fileSubType] = file

    def parseVBFiles(self):
        for vbFileType in self.vbFiles:
            file = self.vbFiles[vbFileType]
            self.vbFileContent[vbFileType] = AGR.FileService.readBinary(file)     

        if (AGR.IniKeywords.Blend not in self.vbFileContent):
            return
        
        # assumption that a mod will have a Blend.buf file and the Blend per vertex is 32 bytes
        #   is generally true for character mods
        blendSizePerVertex = AGR.BufElementTypes.BlendIndicesIntRGBA.value.size + AGR.BufElementTypes.BlendWeightFloatRGBA.value.size
        self.numOfVertices = len(self.vbFileContent[AGR.IniKeywords.Blend]) / blendSizePerVertex

        self.vbSizePerVertex[AGR.IniKeywords.Blend] = blendSizePerVertex

        if (AGR.IniKeywords.Position in self.vbFileContent):
            self.vbSizePerVertex[AGR.IniKeywords.Position] = len(self.vbFileContent[AGR.IniKeywords.Position]) / self.numOfVertices

        if (AGR.IniKeywords.Texcoord in self.vbFileContent):
            self.vbSizePerVertex[AGR.IniKeywords.Texcoord] = len(self.vbFileContent[AGR.IniKeywords.Texcoord]) / self.numOfVertices

    def parseIBFiles(self):
        ibSizePerTriangle = AGR.BufDataTypes.UInt32.value.size * 3
        for ibFileType in self.ibFiles:
            file = self.ibFiles[ibFileType]
            fileContent = AGR.FileService.readBinary(file) 

            self.ibFileContent[ibFileType] = fileContent
            self.numOfTriangles[ibFileType] = len(fileContent) / ibSizePerTriangle

    def parse(self):
        self.parseVBFiles()
        self.parseIBFiles()

    def getVBSummary(self) -> str:
        result = f"Number of vertices: {self.numOfVertices}\n"

        if (AGR.IniKeywords.Position in self.vbFileContent):
            result += f"Position.buf size: {len(self.vbFileContent[AGR.IniKeywords.Position])} bytes\n"
            result += f"Size of position per vertex: {self.vbSizePerVertex[AGR.IniKeywords.Position]} bytes\n"

        if (AGR.IniKeywords.Blend in self.vbFileContent):
            result += f"Blend.buf size: {len(self.vbFileContent[AGR.IniKeywords.Blend])} bytes\n"
            result += f"Size of Blend per vertex: {self.vbSizePerVertex[AGR.IniKeywords.Blend]} bytes\n"

        if (AGR.IniKeywords.Texcoord in self.vbFileContent):
            result += f"Texcoord.buf size: {len(self.vbFileContent[AGR.IniKeywords.Texcoord])} bytes\n"
            result += f"Size of Texcoord per vertex: {self.vbSizePerVertex[AGR.IniKeywords.Texcoord]} bytes\n"

        return result

    def getIBSummary(self) -> str:
        result = ""
        ibSizes = []
        numOfTriangles = []
        triangleTotalSize = 0

        for ibFileType in self.ibFileContent:
            ibSizes.append(f"{ibFileType}: {len(self.ibFileContent[ibFileType])} bytes")
            currentNumOfTriangles = self.numOfTriangles[ibFileType]

            numOfTriangles.append(f"{ibFileType}: {currentNumOfTriangles} triangles")
            triangleTotalSize += currentNumOfTriangles

        if (ibSizes):
            ibSizes = ', '.join(ibSizes)
            ibNumOfTriangles = ", ".join(numOfTriangles)
            result += f".ib sizes: {ibSizes}\n"
            result += f".ib triangles: total: {triangleTotalSize}, {ibNumOfTriangles}\n"

        return result

    def getSummary(self) -> str:
        heading = AGR.Heading(sideLen = 10, sideChar = "=")

        result = f"{heading.close()}\n\n"
        result += f"Subfolder: {self.folder}\n\n"
        result += f"{self.getIBSummary()}\n"
        result += self.getVBSummary()
        result += f"\n{heading.close()}"
        return result

    def analyze(self):
        files = AGR.FileService.getFiles(path = self.folder)
        self.groupFiles(files)

        if (AGR.IniKeywords.Blend not in self.vbFiles):
            raise FileNotFoundError(f"Missing Blend.buf file at {self.folder}")
        
        self.parse()
        result = self.getSummary()
        print(result)

<br>
<br>

## File Setup
Ensure the folder paths to the mods are correctly set for the following constants:

- **ModFolders**

In [4]:
ModFolders = [
    r"../../../Data/Mod Downloads/GI/Kaeya/4_0",
    r"../../../Data/Mod Downloads/GI/KaeyaSailwind/4_0",
]

<br>
<br>

## Run the Analyzer
The code block below analyzes the model binary files for the specified mods

In [5]:
for folder in ModFolders:
    modAnalyzer = ModAnalyzer(folder)
    modAnalyzer.analyze()


Subfolder: ../../../Data/Mod Downloads/GI/Kaeya/4_0

.ib sizes: body: 159012 bytes, dress: 1512 bytes, head: 30384 bytes
.ib triangles: total: 15909.0, body: 13251.0 triangles, dress: 126.0 triangles, head: 2532.0 triangles

Number of vertices: 14711.0
Position.buf size: 588440 bytes
Size of position per vertex: 40.0 bytes
Blend.buf size: 470752 bytes
Size of Blend per vertex: 32 bytes
Texcoord.buf size: 294220 bytes
Size of Texcoord per vertex: 20.0 bytes


Subfolder: ../../../Data/Mod Downloads/GI/KaeyaSailwind/4_0

.ib sizes: body: 214920 bytes, dress: 9924 bytes, head: 92436 bytes
.ib triangles: total: 26440.0, body: 17910.0 triangles, dress: 827.0 triangles, head: 7703.0 triangles

Number of vertices: 21365.0
Position.buf size: 854600 bytes
Size of position per vertex: 40.0 bytes
Blend.buf size: 683680 bytes
Size of Blend per vertex: 32 bytes
Texcoord.buf size: 427300 bytes
Size of Texcoord per vertex: 20.0 bytes

